In [2]:
# ============================================================
# STEP 3 RAG - COMPLETE SETUP
# Run this cell FIRST
# ============================================================

!pip install -q langchain-text-splitters chromadb sentence-transformers

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions


# ------------------------------------------------------------
# Documents from the notebook
# ------------------------------------------------------------

documents = {
    "ai_overview.txt": """
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",

    "embeddings_in_practice.txt": """
Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}


# ------------------------------------------------------------
# Create text splitter
# ------------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)


# ------------------------------------------------------------
# Create ChromaDB
# ------------------------------------------------------------

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Use a unique collection name so we don't get conflicts
collection = client.get_or_create_collection(
    name="rag_step3_collection",
    embedding_function=hf_embedder
)


# ------------------------------------------------------------
# Add the two original documents
# ------------------------------------------------------------

# If collection already has data, create a fresh collection
if collection.count() > 0:
    client.delete_collection("rag_step3_collection")

    collection = client.create_collection(
        name="rag_step3_collection",
        embedding_function=hf_embedder
    )


for source, text in documents.items():

    doc_chunks = splitter.split_text(text)

    collection.add(
        documents=doc_chunks,
        metadatas=[
            {
                "source": source,
                "chunk_index": i
            }
            for i in range(len(doc_chunks))
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(doc_chunks))
        ]
    )


print("Setup completed successfully!")
print("Total chunks stored:", collection.count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup completed successfully!
Total chunks stored: 13


Exercise 1 — Lower threshold to 0.1

In [3]:
# ============================================================
# EXERCISE 1
# Lower similarity threshold to 0.1
# ============================================================

def distance_to_similarity(distance):
    return 1 - distance


def retrieve_with_threshold(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.1
):

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    filtered = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:

            filtered.append({
                "text": doc,
                "metadata": meta,
                "similarity": similarity
            })

    return filtered


# Chocolate cake query
unrelated_matches = retrieve_with_threshold(
    "What is the best recipe for chocolate cake?",
    collection,
    similarity_threshold=0.1
)


print("Similarity threshold:", 0.1)
print("Chunks passed:", len(unrelated_matches))

for match in unrelated_matches:

    print("\nSimilarity:", round(match["similarity"], 4))
    print("Source:", match["metadata"]["source"])
    print("Text:")
    print(match["text"])
    print("-" * 60)

Similarity threshold: 0.1
Chunks passed: 0


Exercise 2 — Add max_chunks based on total character length

In [4]:
# ============================================================
# EXERCISE 2
# Add max_chunks character limit
# ============================================================

def retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    source_filter=None,
    max_chunks=500
):

    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    # Optional metadata filter
    if source_filter:
        query_kwargs["where"] = {
            "source": source_filter
        }

    results = collection.query(**query_kwargs)

    context_chunks = []
    total_characters = 0

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        similarity = distance_to_similarity(distance)

        # Ignore weak matches
        if similarity < similarity_threshold:
            continue

        # Check maximum character limit
        if total_characters + len(doc) > max_chunks:
            break

        context_chunks.append({
            "text": doc,
            "source": meta["source"],
            "chunk_index": meta["chunk_index"],
            "similarity": round(similarity, 4)
        })

        total_characters += len(doc)

    return context_chunks


# ------------------------------------------------------------
# Test the function
# ------------------------------------------------------------

context = retrieve_context(
    "What is machine learning?",
    collection,
    n_results=5,
    similarity_threshold=0.25,
    max_chunks=500
)


print("Chunks returned:", len(context))

total = 0

for c in context:

    print("\nSimilarity:", c["similarity"])
    print("Source:", c["source"])
    print("Chunk:", c["chunk_index"])
    print("Text:")
    print(c["text"])

    total += len(c["text"])

    print("-" * 60)


print("\nTotal characters returned:", total)
print("Maximum allowed:", 500)

Chunks returned: 2

Similarity: 0.7744
Source: ai_overview.txt
Chunk: 3
Text:
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.
------------------------------------------------------------

Similarity: 0.5253
Source: ai_overview.txt
Chunk: 4
Text:
Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
------------------------------------------------------------

Total characters returned: 429
Maximum allowed: 500


Exercise 3

In [5]:
# ============================================================
# EXERCISE 3
# Add a third document
# ============================================================

third_document = """
Python and Artificial Intelligence

Python is one of the most popular programming languages for
Artificial Intelligence and Machine Learning.

Python provides many useful libraries such as NumPy, Pandas,
Scikit-learn and TensorFlow.

Scikit-learn provides tools for machine learning algorithms,
while TensorFlow can be used to build and train neural networks.

Python is also widely used to build AI applications and
data processing pipelines.
"""


# ------------------------------------------------------------
# Split third document
# ------------------------------------------------------------

third_chunks = splitter.split_text(third_document)


# ------------------------------------------------------------
# Create metadata
# ------------------------------------------------------------

third_metadatas = [
    {
        "source": "python_ai.txt",
        "chunk_index": i
    }
    for i in range(len(third_chunks))
]


# ------------------------------------------------------------
# Create unique IDs
# ------------------------------------------------------------

third_ids = [
    f"python_ai_chunk_{i}"
    for i in range(len(third_chunks))
]


# ------------------------------------------------------------
# Add to ChromaDB
# ------------------------------------------------------------

collection.add(
    documents=third_chunks,
    metadatas=third_metadatas,
    ids=third_ids
)


print("Third document chunks added:", len(third_chunks))
print("Total chunks in collection:", collection.count())


# ------------------------------------------------------------
# Test retrieval
# ------------------------------------------------------------

query = "What Python libraries are used for machine learning?"

context = retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    max_chunks=1000
)


print("\nQuery:", query)
print("\nRetrieved context:")
print("=" * 60)


for c in context:

    print("\nSimilarity:", c["similarity"])
    print("Source:", c["source"])
    print("Chunk:", c["chunk_index"])
    print("Text:")
    print(c["text"])

    print("-" * 60)

Third document chunks added: 2
Total chunks in collection: 15

Query: What Python libraries are used for machine learning?

Retrieved context:

Similarity: 0.7362
Source: python_ai.txt
Chunk: 0
Text:
Python and Artificial Intelligence

Python is one of the most popular programming languages for
Artificial Intelligence and Machine Learning.

Python provides many useful libraries such as NumPy, Pandas,
Scikit-learn and TensorFlow.
------------------------------------------------------------

Similarity: 0.7312
Source: python_ai.txt
Chunk: 1
Text:
Scikit-learn provides tools for machine learning algorithms,
while TensorFlow can be used to build and train neural networks.

Python is also widely used to build AI applications and
data processing pipelines.
------------------------------------------------------------

Similarity: 0.4848
Source: ai_overview.txt
Chunk: 3
Text:
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead